# Chapter 14: Checkpointing

[Read this chapter online](https://jackluu.io/book/section-4-training/ch14-checkpointing/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch14-checkpointing.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 14: Checkpointing

![You are here in the big picture](../assets/diagrams/ch14-where-we-are.png){ width="756" }
*Figure 14.1: Where we are: we have trained the model and now we save its weights.*

Training a language model takes time. Once the model learns from the data, you need to save its knowledge so you can use it later without retraining. In this chapter you will:

- Save a trained model to a file.
- Load a saved model back into memory.
- Switch the model from training mode to evaluation mode.

**Words to Know**
    - **Checkpoint**: a file containing the saved state (weights and configuration) of a model at a specific point in training.
    - **State Dict**: a PyTorch dictionary that maps each layer of the model to its learned weights.
    - **Dropout**: A technique that randomly turns off some neurons during training to prevent memorizing the data.

## Theory

### The Checkpoint File

![A trained model saves its weights and configuration to model.pt; later, an empty model is built from the configuration and restored with the saved weights.](../assets/diagrams/ch14-save-load.png){ width="498" }
*Figure 14.2: The save and load round trip: the model's structure and its learned weights are both stored in the checkpoint.*

A checkpoint is the model's save file. When you train a model, you adjust its weights (the parameters). These learned parameters are stored in a dictionary called a `state_dict`. 

However, the weights alone are not enough. If you close your program and come back tomorrow, PyTorch will not know how many layers or attention heads your model has. To bring the model back to life, you must save both its configuration (the architecture skeleton) and its state dictionary (the learned weights). We also save the current training step and validation loss so we know how well the model performed.

### Restoring the Model

Loading a checkpoint happens in three steps:

1. Load the saved dictionary from the file on disk.
2. Build an empty model using the saved configuration.
3. Pour the saved weights into the empty model.

Once the model is loaded, you must call `model.eval()`. During training, neural networks often use a technique called dropout, which randomly turns off some neurons to prevent the model from memorizing the data. Calling `model.eval()` turns off dropout, ensuring that all neurons are active and the model gives reliable, deterministic answers when generating text.

![Training mode with some neurons off versus eval mode with all neurons active](../assets/diagrams/ch14-eval-mode.png){ width="458" }
*Figure 14.3: In evaluation mode, all neurons are active and the model is ready to generate text deterministically.*

As the "where we are" map shows, saving the checkpoint captures the model after the training loop, preparing it to generate new text in the final stage.

**In Business**
    Think of our house-style assistant. The training process analyzed your company's archive to learn its voice. If the server restarts, you don't want to re-read the entire archive. A checkpoint saves that learned company voice to a tiny file that you can load instantly on any machine.

## Code

We use the PyTorch `torch.load()` function to read our checkpoint file, and `model.load_state_dict()` to apply the weights.

```python
checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=False
    )
    step     = checkpoint["step"]
    val_loss = checkpoint["val_loss"]
    cfg      = checkpoint["gpt_cfg"]

    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])

    # Set to evaluation mode (disables dropout, making outputs deterministic)
    model.eval()
```

Let's see what happens when we load the checkpoint created at the end of the previous chapter.

```python
$ python src/ch13_checkpoint.py
--- 2. Load the checkpoint ---
Loading checkpoint from: checkpoints/model.pt
  Trained for  : 3000 steps
  Val loss     : 1.7228
  Model config : 4 layers, 128 embd, 4 heads

--- 3. Rebuild the model from the checkpoint ---

Model rebuilt successfully: 824,832 parameters loaded

--- 4. Verify the model works ---
Input shape : torch.Size([1, 10])
Output shape: torch.Size([1, 10, 65])   (looks good!)

--- 5. Show checkpoint file size ---

Checkpoint file size: 4.18 MB
(Small enough to share by email!)

Checkpointing done! Ready for Chapter 15.
```

![Code flow: load file, extract dictionary, load into empty model](../assets/diagrams/ch14-code-flow.png){ width="758" }
*Figure 14.4: How the checkpoint loading code flows: from file to ready model.*

**What just happened:**

- Line 1 loaded the checkpoint `model.pt` from disk.
- Line 8 built an empty `GPT` model using the loaded configuration.
- Line 9 populated the model with the 824,832 learned parameters.
- Line 12 switched the model to evaluation mode.
- We verified the checkpoint file is tiny (just over 4 megabytes).

### Shape Check

Table 14.1 shows the tensor shapes when verifying the loaded model.

**Table 14.1:** Input and output shapes for the restored model.

| Tensor | Shape | What it means |
|--------|-------|---------------|
| `dummy_ids` | `[1, 10]` | 1 sequence of 10 token IDs. |
| `logits` | `[1, 10, 65]` | 65 vocabulary scores for each of the 10 positions. |

## Try It

**Try It**
    You can inspect the checkpoint directly by writing a small script to load the file.
    
    ```python title="src/examples/ch14_inspect_checkpoint.py (excerpt)" linenums="1" hl_lines="9 10"
    # ...
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
    )
    # ...
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    print("Keys in checkpoint:", list(checkpoint.keys()))
    print("Training step:", checkpoint.get("step"))
    ```
    
    Lines 9 and 10 print the keys and the training step from the loaded checkpoint dictionary.
    
    ```console title="Terminal"
    $ python src/examples/ch14_inspect_checkpoint.py
    Keys in checkpoint: ['model_state', 'gpt_cfg', 'step', 'val_loss']
    Training step: 3000

    ```

## Key Takeaways

- A checkpoint saves the model's configuration and learned weights to a file.
- You rebuild the model by creating an empty network with the configuration, then loading the weights with `load_state_dict`.
- Always call `model.eval()` after loading a model to turn off training features like dropout.
- A model with 825,000 parameters takes up only about 4 MB of disk space.

## Check Your Understanding

1. Why do we need to save the model's configuration in the checkpoint along with the weights?
2. What happens if you forget to call `model.eval()` before generating text?
3. What PyTorch function do we use to apply the saved weights to the newly built model?


## Further Reading

**The library you are typing into.** The design argument behind the tool this book uses: write the model as ordinary Python that runs line by line, so you can print a tensor or stop in a debugger, and still get the speed of compiled code underneath. It is the reason the code in this book can be read top to bottom and still trains a real model.

<div class="refs" markdown>

Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., Killeen, T., Lin, Z., Gimelshein, N., Antiga, L., Desmaison, A., Köpf, A., Yang, E., DeVito, Z., Raison, M., Tejani, A., Chilamkurthy, S., Steiner, B., Fang, L., ... Chintala, S. (2019). *PyTorch: An imperative style, high-performance deep learning library* (arXiv:1912.01703). arXiv. https://doi.org/10.48550/arXiv.1912.01703

</div>

---

### `src/ch13_checkpoint.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch13_checkpoint.py"   # a cell has none, and the file uses it to find the text

"""
Save and load model checkpoints.
This file belongs to Chapter 14.
Run: python src/ch13_checkpoint.py
"""
import os
import torch

import sys

from src.utils.config import GPTConfig
from src.ch09_gpt_model import GPT

# Settings
CHECKPOINT_PATH = "checkpoints/model.pt"

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 14: Saving and Loading Checkpoints\n")

    print("--- 1. Check if a checkpoint exists ---")
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"\nNo checkpoint found at {CHECKPOINT_PATH}.")
        print("Creating a dummy model to demonstrate save/load...")
        cfg = GPTConfig()
        model = GPT(cfg)
        os.makedirs("checkpoints", exist_ok=True)
        torch.save({
            "model_state": model.state_dict(),
            "gpt_cfg"    : cfg,
            "step"       : 0,
            "val_loss"   : float("inf"),
        }, CHECKPOINT_PATH)
        print(f"Dummy checkpoint saved to {CHECKPOINT_PATH}")

    print("\n--- 2. Load the checkpoint ---")
    print(f"Loading checkpoint from: {CHECKPOINT_PATH}")

    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=False
    )
    step     = checkpoint["step"]
    val_loss = checkpoint["val_loss"]
    cfg      = checkpoint["gpt_cfg"]

    print(f"  Trained for  : {step} steps")
    print(f"  Val loss     : {val_loss:.4f}")
    print(f"  Model config : {cfg.n_layers} layers, {cfg.n_embd} embd, "
          f"{cfg.n_heads} heads")

    print("\n--- 3. Rebuild the model from the checkpoint ---")
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])

    # Set to evaluation mode (disables dropout, making outputs deterministic)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel rebuilt successfully: {total_params:,} parameters loaded")

    print("\n--- 4. Verify the model works ---")
    dummy_ids = torch.zeros((1, 10), dtype=torch.long)
    with torch.no_grad():
        logits = model(dummy_ids)

    print(f"Input shape : {dummy_ids.shape}")
    print(f"Output shape: {logits.shape}   (looks good!)")

    print("\n--- 5. Show checkpoint file size ---")
    size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)
    print(f"\nCheckpoint file size: {size_mb:.2f} MB")
    print("(Small enough to share by email!)")

    print("\nCheckpointing done! Ready for Chapter 15.")

---

### `src/examples/ch14_inspect_checkpoint.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch14_inspect_checkpoint.py"   # a cell has none, and the file uses it to find the text

"""Inspect the contents of the saved PyTorch checkpoint."""
import os, sys
import torch

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

def main():
    torch.manual_seed(42)
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
    )
    
    if not os.path.exists(checkpoint_path):
        print("Checkpoint not found.")
        return
        
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    print("Keys in checkpoint:", list(checkpoint.keys()))
    print("Training step:", checkpoint.get("step"))

if __name__ == "__main__":
    main()